In [2]:
%cd PRO506

[Errno 2] No such file or directory: 'PRO506'
/workspace/PRO506


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import StringType, IntegerType, FloatType,BooleanType,DateType,TimestampType
from pyspark.sql import functions as fun
from pyspark.sql import Window
import pandas as pd 

spark = ( SparkSession.builder.appName("PRO506").master("spark://spark-master:7077").getOrCreate())



sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 10:03:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
main_schema = StructType([
    StructField("row_id",StringType(),True),
    StructField("click_datetime",TimestampType(),True),
    StructField("time_to_next_click",FloatType(),True),
    StructField("movie_title",StringType(),True),
    StructField("movie_genres",StringType(),True),
    StructField("relase_date",DateType(),True),
    StructField("title_id",StringType(),True),
    StructField("user_id",StringType(),True),
])

df = (spark.read.format("csv")
      .option("header","true")
      .schema(main_schema)
      .option("sep",",")
      .load("vodclickstream_uk_movies_03.csv"))

df.show(5)

+------+-------------------+------------------+--------------------+--------------------+-----------+----------+----------+
|row_id|     click_datetime|time_to_next_click|         movie_title|        movie_genres|relase_date|  title_id|   user_id|
+------+-------------------+------------------+--------------------+--------------------+-----------+----------+----------+
| 58773|2017-01-01 01:15:09|               0.0|Angus, Thongs and...|Comedy, Drama, Ro...| 2008-07-25|26bd5987e8|1dea19f6fe|
| 58774|2017-01-01 13:56:02|               0.0|The Curse of Slee...|Fantasy, Horror, ...| 2016-06-02|f26ed2675e|544dcbc510|
| 58775|2017-01-01 15:17:47|           10530.0|   London Has Fallen|    Action, Thriller| 2016-03-04|f77e500e7a|7cbcc791bf|
| 58776|2017-01-01 16:04:13|              49.0|            Vendetta|       Action, Drama| 2015-06-12|c74aec7673|ebf43c36b6|
| 58777|2017-01-01 19:16:37|               0.0|The SpongeBob Squ...|Animation, Action...| 2004-11-19|a80d6fc2aa|a57c992287|
+------+

In [5]:
ventana = (Window.partitionBy("User_id")).orderBy("click_datetime")

df = df.withColumn(
    "calculated_time_to_next",
    ((fun.lag("click_datetime").over(ventana)).cast("long") - fun.col("click_datetime").cast("long"))*(-1)
)
df.show(5)

[Stage 1:=======>                                                   (1 + 7) / 8]

+------+-------------------+------------------+--------------------+--------------------+-----------+----------+----------+-----------------------+
|row_id|     click_datetime|time_to_next_click|         movie_title|        movie_genres|relase_date|  title_id|   user_id|calculated_time_to_next|
+------+-------------------+------------------+--------------------+--------------------+-----------+----------+----------+-----------------------+
|139643|2017-05-19 20:21:43|               0.0|                XOXO|        Drama, Music| 2016-08-26|7369676dec|0006ea6b5c|                   NULL|
|140442|2017-05-20 21:54:34|               0.0|            Hot Fuzz|Action, Comedy, M...| 2007-04-20|6467fee6b6|0006ea6b5c|                  91971|
|144717|2017-05-26 18:38:01|               0.0|         War Machine|  Comedy, Drama, War| 2017-05-26|0f3b137f4e|0006ea6b5c|                 506607|
|144301|2017-05-26 23:31:46|               0.0|          Apocalypto|Action, Adventure...| 2006-12-08|40dd7bf1f9|

In [6]:
df = df.withColumn(
    "es_zaping",
    fun.when(((fun.lag("click_datetime").over(ventana)).cast("long") - fun.col("click_datetime").cast("long"))*(-1) > 300 , "No").otherwise("Si") 
)
df.show(5)

[Stage 4:=======>                                                   (1 + 7) / 8]

+------+-------------------+------------------+--------------------+--------------------+-----------+----------+----------+-----------------------+---------+
|row_id|     click_datetime|time_to_next_click|         movie_title|        movie_genres|relase_date|  title_id|   user_id|calculated_time_to_next|es_zaping|
+------+-------------------+------------------+--------------------+--------------------+-----------+----------+----------+-----------------------+---------+
|139643|2017-05-19 20:21:43|               0.0|                XOXO|        Drama, Music| 2016-08-26|7369676dec|0006ea6b5c|                   NULL|       Si|
|140442|2017-05-20 21:54:34|               0.0|            Hot Fuzz|Action, Comedy, M...| 2007-04-20|6467fee6b6|0006ea6b5c|                  91971|       No|
|144717|2017-05-26 18:38:01|               0.0|         War Machine|  Comedy, Drama, War| 2017-05-26|0f3b137f4e|0006ea6b5c|                 506607|       No|
|144301|2017-05-26 23:31:46|               0.0|     

In [7]:
df = df.withColumn("click_date", fun.split(fun.col("click_datetime")," ")[0])

ventana = (Window.partitionBy("User_id","click_date")).orderBy("click_datetime")

df = df.withColumn("row_number", fun.row_number().over(ventana)).orderBy("row_number")

df.filter(fun.col("row_number") >= 5).show(5)



[Stage 9:>                                                          (0 + 8) / 9]

+------+-------------------+------------------+-----------+--------------------+-----------+----------+----------+-----------------------+---------+----------+----------+
|row_id|     click_datetime|time_to_next_click|movie_title|        movie_genres|relase_date|  title_id|   user_id|calculated_time_to_next|es_zaping|click_date|row_number|
+------+-------------------+------------------+-----------+--------------------+-----------+----------+----------+-----------------------+---------+----------+----------+
|661532|2019-04-01 13:09:14|            7758.0|  Black '47|       Action, Drama| 2018-09-28|3641f15b80|004e33f215|                    513|       No|2019-04-01|         5|
|381700|2018-03-30 00:53:10|               0.0|   R.I.P.D.|Action, Adventure...| 2013-07-19|7ae6da3782|02cd2456b2|                    130|       Si|2018-03-30|         5|
|121362|2017-04-26 12:31:08|            3000.0|World War Z|Action, Adventure...| 2013-06-21|30c17c9520|0082efd082|                   2400|       

In [8]:
ventana = Window.partitionBy("user_id","title_id")

df = df.withColumn(
    "veces_vista_por_usuario",
    fun.count("user_id").over(ventana)
)

df.filter(fun.col("veces_vista_por_usuario") > 3).show(25)

[Stage 17:===========================================>              (3 + 1) / 4]

+------+-------------------+------------------+--------------------+--------------------+-----------+----------+----------+-----------------------+---------+----------+----------+-----------------------+
|row_id|     click_datetime|time_to_next_click|         movie_title|        movie_genres|relase_date|  title_id|   user_id|calculated_time_to_next|es_zaping|click_date|row_number|veces_vista_por_usuario|
+------+-------------------+------------------+--------------------+--------------------+-----------+----------+----------+-----------------------+---------+----------+----------+-----------------------+
|578906|2018-12-30 22:05:13|               0.0|Black Mirror: Ban...|Drama, Mystery, S...| 2018-12-28|e847f14da5|000296842d|                   NULL|       Si|2018-12-30|         1|                      8|
|579565|2018-12-31 01:40:54|             907.0|Black Mirror: Ban...|Drama, Mystery, S...| 2018-12-28|e847f14da5|000296842d|                  12941|       No|2018-12-31|         1|     

26/03/05 10:03:44 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
